# Data acquisition and audit

This notebook loads BFRD and a deterministic BanglishRev text sample, records source provenance, removes invalid records, creates duplicate-aware gold splits, and writes the stage-one artifacts.


In [ ]:
%pip install -q -r requirements.txt

import hashlib
import heapq
import ijson
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_url
from pyspark.ml.feature import HashingTF, MinHashLSH
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ArrayType, DoubleType, IntegerType, LongType, StringType, StructField, StructType
)

SEED = 42
BANGLISH_SAMPLE_SIZE = int(os.getenv("BANGLISH_SAMPLE_SIZE", "100000"))
BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
for directory in (PROCESSED_DIR, AUDIT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_Acquisition")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [ ]:
BFRD_REPO = "shawon95/Bengali-Fake-Review-Dataset"
BANGLISH_REPO = "BanglishRev/bangla-english-and-code-mixed-ecommerce-review-dataset"

api = HfApi()
bfrd_info = api.dataset_info(BFRD_REPO)
banglish_info = api.dataset_info(BANGLISH_REPO)
source_versions = {
    "BFRD": bfrd_info.sha,
    "BanglishRev": banglish_info.sha,
}

bfrd_hf = load_dataset(BFRD_REPO, split="train", revision=bfrd_info.sha)
bfrd_pdf = bfrd_hf.to_pandas()
column_lookup = {str(column).strip().lower(): column for column in bfrd_pdf.columns}
if "review" not in column_lookup or "label" not in column_lookup:
    raise ValueError(f"Unexpected BFRD columns: {list(bfrd_pdf.columns)}")

bfrd_pdf = bfrd_pdf[[column_lookup["review"], column_lookup["label"]]].copy()
bfrd_pdf.columns = ["review", "label"]
bfrd_pdf["review"] = bfrd_pdf["review"].astype("string")
bfrd_pdf["label"] = pd.to_numeric(bfrd_pdf["label"], errors="coerce")

bfrd_schema = StructType([
    StructField("review", StringType(), True),
    StructField("label", IntegerType(), True),
])
bfrd_raw = spark.createDataFrame(
    [(None if pd.isna(row.review) else str(row.review),
      None if pd.isna(row.label) else int(row.label))
     for row in bfrd_pdf.itertuples(index=False)],
    schema=bfrd_schema,
)

raw_bfrd_count = bfrd_raw.count()
raw_bfrd_labels = {
    int(row["label"]): int(row["count"])
    for row in bfrd_raw.groupBy("label").count().collect()
    if row["label"] is not None
}
if raw_bfrd_count != 9049:
    raise ValueError(f"BFRD row-count mismatch: expected 9049, found {raw_bfrd_count}")
if raw_bfrd_labels != {0: 1339, 1: 7710}:
    raise ValueError(f"BFRD label-count mismatch: {raw_bfrd_labels}")

bfrd_valid = (
    bfrd_raw
    .withColumn("review", F.trim(F.col("review")))
    .filter(F.col("review").isNotNull() & (F.length("review") > 0))
    .filter(F.col("label").isin(0, 1))
    .withColumn("review_group_id", F.sha2(F.col("review"), 256))
)

exact_groups = bfrd_valid.groupBy("review_group_id").agg(
    F.count("*").alias("exact_dup_cluster_size"),
    F.countDistinct("label").alias("label_count"),
)
conflicting_groups = exact_groups.filter(F.col("label_count") > 1)
conflicting_ids = conflicting_groups.select("review_group_id")
bfrd_quarantine = bfrd_valid.join(conflicting_ids, "review_group_id", "inner")
bfrd_quarantine.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "bfrd_quarantine.parquet")
)

bfrd_clean = (
    bfrd_valid.join(conflicting_ids, "review_group_id", "left_anti")
    .join(exact_groups.select("review_group_id", "exact_dup_cluster_size"), "review_group_id")
    .dropDuplicates(["review_group_id"])
    .withColumn("review_id", F.col("review_group_id"))
    .withColumn("is_exact_duplicate", (F.col("exact_dup_cluster_size") > 1).cast("int"))
)
clean_bfrd_labels = {
    int(row["label"]): int(row["count"])
    for row in bfrd_clean.groupBy("label").count().collect()
}


In [ ]:
def value_case_insensitive(record, *names):
    lookup = {str(key).strip().lower(): value for key, value in record.items()}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


def iter_banglish_products():
    json_files = [
        sibling.rfilename for sibling in banglish_info.siblings
        if sibling.rfilename.lower().endswith(".json")
    ]
    if not json_files:
        raise RuntimeError("No BanglishRev JSON file was found")
    preferred = sorted(
        json_files,
        key=lambda name: ("review" not in name.lower(), len(name), name),
    )[0]
    data_url = hf_hub_url(
        BANGLISH_REPO, preferred, repo_type="dataset", revision=banglish_info.sha
    )
    with requests.get(data_url, stream=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        response.raw.decode_content = True
        yield from ijson.items(response.raw, "item")


reservoir = []
scanned_reviews = 0
sequence = 0
for product in iter_banglish_products():
    product_id = value_case_insensitive(product, "Product ID", "product_id")
    category = value_case_insensitive(product, "Category", "category")
    parent_category = value_case_insensitive(product, "Parent Category", "parent_category")
    reviews = value_case_insensitive(product, "Reviews", "reviews") or []
    if isinstance(reviews, dict):
        reviews = [reviews]

    for review_object in reviews:
        if not isinstance(review_object, dict):
            continue
        review = value_case_insensitive(
            review_object, "Review Content", "review_content", "review_text", "review"
        )
        if review is None or not str(review).strip():
            continue

        row = {
            "review": str(review).strip(),
            "rating": value_case_insensitive(review_object, "Current Rating", "rating"),
            "product_id": None if product_id is None else str(product_id),
            "review_date": value_case_insensitive(review_object, "Review Date", "review_date"),
            "date_bought": value_case_insensitive(review_object, "Date Bought", "date_bought"),
            "likes": value_case_insensitive(review_object, "Likes", "likes"),
            "dislikes": value_case_insensitive(review_object, "Dislikes", "dislikes"),
            "reply": value_case_insensitive(review_object, "Reply", "reply"),
            "category": None if category is None else str(category),
            "parent_category": None if parent_category is None else str(parent_category),
        }
        key = "|".join([
            row["product_id"] or "",
            str(row["review_date"] or ""),
            row["review"],
        ])
        score = int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16)
        sequence += 1
        scanned_reviews += 1
        candidate = (-score, sequence, row)
        if len(reservoir) < BANGLISH_SAMPLE_SIZE:
            heapq.heappush(reservoir, candidate)
        elif score < -reservoir[0][0]:
            heapq.heapreplace(reservoir, candidate)

if not reservoir:
    raise RuntimeError("BanglishRev produced no written reviews")

sample_rows = [item[2] for item in sorted(reservoir, key=lambda item: -item[0])]
sample_digest = hashlib.sha256()
for row in sample_rows:
    sample_digest.update(
        json.dumps(row, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
    )

def as_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

banglish_schema = StructType([
    StructField("review", StringType(), False),
    StructField("rating", DoubleType(), True),
    StructField("product_id", StringType(), True),
    StructField("review_date", StringType(), True),
    StructField("date_bought", StringType(), True),
    StructField("likes", LongType(), True),
    StructField("dislikes", LongType(), True),
    StructField("reply", StringType(), True),
    StructField("category", StringType(), True),
    StructField("parent_category", StringType(), True),
])

banglish_raw = spark.createDataFrame([
    (
        row["review"],
        as_float(row["rating"]),
        row["product_id"],
        None if row["review_date"] is None else str(row["review_date"]),
        None if row["date_bought"] is None else str(row["date_bought"]),
        as_int(row["likes"]),
        as_int(row["dislikes"]),
        None if row["reply"] is None else str(row["reply"]),
        row["category"],
        row["parent_category"],
    )
    for row in sample_rows
], schema=banglish_schema)

banglish_with_ids = (
    banglish_raw
    .withColumn("review_group_id", F.sha2(F.col("review"), 256))
    .withColumn(
        "review_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.coalesce(F.col("product_id"), F.lit("")),
                F.coalesce(F.col("review_date"), F.lit("")),
                F.col("review"),
            ),
            256,
        ),
    )
)

banglish_exact = banglish_with_ids.groupBy("review_group_id").agg(
    F.count("*").alias("exact_dup_cluster_size")
)
banglish_clean = (
    banglish_with_ids
    .join(banglish_exact, "review_group_id")
    .dropDuplicates(["review_group_id"])
    .withColumn("is_exact_duplicate", (F.col("exact_dup_cluster_size") > 1).cast("int"))
    .withColumn("label", F.lit(None).cast("int"))
    .withColumn("source_dataset", F.lit("BanglishRev"))
    .withColumn("source_split", F.lit("unlabeled_pool"))
    .withColumn("label_source", F.lit("unlabeled"))
    .withColumn("confidence_score", F.lit(None).cast("double"))
)

banglish_clean.write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "banglish_unlabeled.parquet")
)


In [ ]:
@F.udf(ArrayType(StringType()))
def split_shingles(text):
    tokens = [token for token in str(text or "").lower().split() if token]
    if len(tokens) < 3:
        return ["__short__:" + "_".join(tokens)] if tokens else ["__empty__"]
    return ["_".join(tokens[index:index + 3]) for index in range(len(tokens) - 2)]


gold_shingles = bfrd_clean.withColumn("split_shingles", split_shingles("review"))
hashing = HashingTF(
    inputCol="split_shingles",
    outputCol="split_shingle_vector",
    numFeatures=16384,
    binary=True,
)
gold_vectors = hashing.transform(gold_shingles)
split_lsh = MinHashLSH(
    inputCol="split_shingle_vector",
    outputCol="split_hashes",
    numHashTables=5,
    seed=SEED,
).fit(gold_vectors)

near_pairs = (
    split_lsh.approxSimilarityJoin(
        gold_vectors,
        gold_vectors,
        threshold=0.30,
        distCol="jaccard_distance",
    )
    .filter(F.col("datasetA.review_id") < F.col("datasetB.review_id"))
    .select(
        F.col("datasetA.review_id").alias("left_id"),
        F.col("datasetB.review_id").alias("right_id"),
    )
    .dropDuplicates()
    .collect()
)

gold_ids = [row["review_id"] for row in bfrd_clean.select("review_id").collect()]
parent = {review_id: review_id for review_id in gold_ids}

def find(review_id):
    while parent[review_id] != review_id:
        parent[review_id] = parent[parent[review_id]]
        review_id = parent[review_id]
    return review_id

def union(left_id, right_id):
    left_root = find(left_id)
    right_root = find(right_id)
    if left_root != right_root:
        parent[max(left_root, right_root)] = min(left_root, right_root)

for pair in near_pairs:
    union(pair["left_id"], pair["right_id"])

group_rows = [(review_id, find(review_id)) for review_id in gold_ids]
group_schema = StructType([
    StructField("review_id", StringType(), False),
    StructField("duplicate_group_id", StringType(), False),
])
group_map = spark.createDataFrame(group_rows, group_schema)
gold_grouped = bfrd_clean.join(group_map, "review_id")

group_counts = (
    gold_grouped.groupBy("duplicate_group_id", "label")
    .count()
    .toPandas()
)
group_table = (
    group_counts.pivot(index="duplicate_group_id", columns="label", values="count")
    .fillna(0)
    .reset_index()
)
for label in (0, 1):
    if label not in group_table.columns:
        group_table[label] = 0

targets = {
    "train": {0: clean_bfrd_labels[0] * 0.70, 1: clean_bfrd_labels[1] * 0.70},
    "validation": {0: clean_bfrd_labels[0] * 0.15, 1: clean_bfrd_labels[1] * 0.15},
    "test": {0: clean_bfrd_labels[0] * 0.15, 1: clean_bfrd_labels[1] * 0.15},
}
assigned = {
    split: {0: 0, 1: 0}
    for split in ("train", "validation", "test")
}

group_table["size"] = group_table[0] + group_table[1]
group_table["order_key"] = group_table["duplicate_group_id"].map(
    lambda value: hashlib.sha256(f"{SEED}|{value}".encode("utf-8")).hexdigest()
)
group_table = group_table.sort_values(
    ["size", "order_key"], ascending=[False, True]
)

assignments = []
for row in group_table.itertuples(index=False):
    counts = {0: int(getattr(row, "_1")), 1: int(getattr(row, "_2"))}
    priorities = {}
    for split in assigned:
        priorities[split] = sum(
            counts[label]
            * max(targets[split][label] - assigned[split][label], 0)
            / max(targets[split][label], 1)
            for label in (0, 1)
        )
    chosen = max(
        ("train", "validation", "test"),
        key=lambda split: (priorities[split], -("train", "validation", "test").index(split)),
    )
    assignments.append((row.duplicate_group_id, chosen))
    for label in (0, 1):
        assigned[chosen][label] += counts[label]

assignment_schema = StructType([
    StructField("duplicate_group_id", StringType(), False),
    StructField("source_split", StringType(), False),
])
assignment_df = spark.createDataFrame(assignments, assignment_schema)

gold_split = (
    gold_grouped.join(assignment_df, "duplicate_group_id")
    .withColumn("source_dataset", F.lit("BFRD"))
    .withColumn("label_source", F.lit("gold"))
    .withColumn("confidence_score", F.lit(None).cast("double"))
)

train = gold_split.filter(F.col("source_split") == "train")
validation = gold_split.filter(F.col("source_split") == "validation")
test = gold_split.filter(F.col("source_split") == "test")

train.write.mode("overwrite").parquet(str(PROCESSED_DIR / "bfrd_train.parquet"))
validation.write.mode("overwrite").parquet(str(PROCESSED_DIR / "bfrd_validation.parquet"))
test.write.mode("overwrite").parquet(str(PROCESSED_DIR / "bfrd_test_locked.parquet"))

overlap_count = (
    gold_split.groupBy("duplicate_group_id")
    .agg(F.countDistinct("source_split").alias("split_count"))
    .filter(F.col("split_count") > 1)
    .count()
)
if overlap_count:
    raise AssertionError("A duplicate group appears in multiple gold splits")


In [ ]:
def dataframe_audit(frame, text_column):
    return {
        "rows": frame.count(),
        "null_text": frame.filter(F.col(text_column).isNull()).count(),
        "blank_text": frame.filter(F.trim(F.col(text_column)) == "").count(),
        "columns": frame.columns,
    }

split_counts = {
    row["source_split"]: {
        int(item["label"]): int(item["count"])
        for item in gold_split.filter(F.col("source_split") == row["source_split"])
        .groupBy("label").count().collect()
    }
    for row in gold_split.select("source_split").distinct().collect()
}

audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source_versions": source_versions,
    "BFRD": {
        "raw": dataframe_audit(bfrd_raw, "review"),
        "raw_label_counts": raw_bfrd_labels,
        "clean_label_counts": clean_bfrd_labels,
        "clean_rows": bfrd_clean.count(),
        "quarantined_conflicting_rows": bfrd_quarantine.count(),
        "near_duplicate_pairs": len(near_pairs),
        "duplicate_groups": gold_grouped.select("duplicate_group_id").distinct().count(),
        "split_label_counts": split_counts,
    },
    "BanglishRev": {
        "written_reviews_scanned": scanned_reviews,
        "requested_sample_size": BANGLISH_SAMPLE_SIZE,
        "sample_rows_before_deduplication": len(sample_rows),
        "clean_rows": banglish_clean.count(),
        "sample_sha256": sample_digest.hexdigest(),
        "sampling_method": "deterministic bottom-k SHA-256 sample over all written reviews",
        "excluded_fields": ["Buyer ID", "Images"],
    },
}

with open(AUDIT_DIR / "acquisition_audit.json", "w", encoding="utf-8") as handle:
    json.dump(audit, handle, ensure_ascii=False, indent=2)

gold_split.createOrReplaceTempView("gold_split")
spark.sql("""
    SELECT source_split, label, COUNT(*) AS rows
    FROM gold_split
    GROUP BY source_split, label
    ORDER BY source_split, label
""").show()

print(json.dumps(audit, ensure_ascii=False, indent=2))
